In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
folder_path = Path("/home/tamires/projects/rpp-aevans-ab/tamires/")
output_path = Path("BrainForecast/inputs/")

## 1. Movie Dataset

In [2]:
df_raw_movie_path = Path("DecomposingMovies/outputs/combined_movies_features.parquet")
df_su_movie_path = Path("DecomposingMovies/outputs/combined_movies_surprise_uncertainty.parquet")
df_raw_movie = pd.read_parquet(folder_path/df_raw_movie_path)
df_su_movie = pd.read_parquet(folder_path/df_su_movie_path)
df_combined = pd.concat([df_raw_movie.drop(columns=['movie']), df_su_movie], axis=1)
df_combined["time_s"] = df_combined.groupby(["movie"]).cumcount() 
df_combined["movie"] = df_combined["movie"].str.replace("_", "", regex=False)
df_combined = df_combined.rename(columns=lambda c: c if c in ["time_s", "movie"] else f"mov_{c}")
#df_combined.to_parquet(folder_path/output_path/'known_stimuli_per_second.parquet')
df_combined[["time_s","movie"]]

,time_s,movie
0,0,12yearsaslave
1,1,12yearsaslave
2,2,12yearsaslave
3,3,12yearsaslave
4,4,12yearsaslave
...,...,...
69986,6098,theusualsuspects
69987,6099,theusualsuspects
69988,6100,theusualsuspects
69989,6101,theusualsuspects


In [4]:
df_raw_movie.columns

Index(['L1_0000', 'L1_0001', 'L1_0002', 'L1_0003', 'L1_0004', 'L1_0005',
       'L1_0006', 'L1_0007', 'L1_0008', 'L1_0009',
       ...
       'narrative_0375', 'narrative_0376', 'narrative_0377', 'narrative_0378',
       'narrative_0379', 'narrative_0380', 'narrative_0381', 'narrative_0382',
       'narrative_0383', 'movie'],
      dtype='object', length=4901)

In [5]:
df_su_movie.columns

Index(['time_s', 'n_frames', 'scene_cut', 'surprise_L1', 'uncertainty_L1',
       'surprise_L2', 'uncertainty_L2', 'surprise_L3', 'uncertainty_L3',
       'surprise_L4', 'uncertainty_L4', 'surprise_semantic',
       'uncertainty_semantic', 'surprise_motion', 'uncertainty_motion',
       'surprise_emotion', 'uncertainty_emotion', 'surprise_faces',
       'uncertainty_faces', 'n_faces_mean', 'n_faces_std',
       'face_coverage_mean', 'face_coverage_std', 'dominant_emotion_idx',
       'dominant_emotion', 'surprise_audio_mel', 'uncertainty_audio_mel',
       'surprise_audio_spec', 'uncertainty_audio_spec', 'surprise_audio_onset',
       'uncertainty_audio_onset', 'surprise_narrative',
       'uncertainty_narrative', 'interaction_L1', 'interaction_L2',
       'interaction_L3', 'interaction_L4', 'interaction_semantic',
       'interaction_emotion', 'interaction_faces', 'interaction_motion',
       'interaction_audio_mel', 'interaction_audio_spec',
       'interaction_audio_onset', 'interac

In [11]:
df_combined['mov_dominant_emotion'] = df_combined['mov_dominant_emotion'].astype('category').cat.codes
df_combined['mov_dominant_emotion']

0        0
1        0
2        0
3        0
4        0
        ..
69986    5
69987    5
69988    5
69989    0
69990    0
Name: mov_dominant_emotion, Length: 69991, dtype: int8

## 2. fMRI activation per second

In [12]:
df_fmri_folder = Path("DecomposingFMRI/outputs/bold_parcels_harvard_oxford/")
base_path = folder_path / df_fmri_folder

dfs = []

for file_path in base_path.glob("*.parquet"):
    
    df = pd.read_parquet(file_path)
    sub_id, movie_name = file_path.stem.split("_", 1)
    sub_id = sub_id.replace("sub-", "")
    #movie_name = movie_name.replace(".parquet", "")

    # add metadata
    df["cohort"] = "ds002837"
    df["sub"] = sub_id
    df["movie"] = movie_name

    dfs.append(df)

# final concatenation
df_all = pd.concat(dfs, axis=0, ignore_index=True)
df_all = df_all.rename(columns=lambda c: c if c in ["time_s", "movie", "cohort", "sub"] else f"b_{c}")
#df_all.to_parquet(folder_path/output_path/'brain_observed_dynamic_per_second.parquet')
df_all

,time_s,b_Left_Frontal_Pole,b_Right_Frontal_Pole,b_Left_Insular_Cortex,b_Right_Insular_Cortex,b_Left_Superior_Frontal_Gyrus,b_Right_Superior_Frontal_Gyrus,b_Left_Middle_Frontal_Gyrus,b_Right_Middle_Frontal_Gyrus,b_Left_Inferior_Frontal_Gyrus_pars_triangularis,...,b_Right_Thalamus,b_Right_Caudate,b_Right_Putamen,b_Right_Pallidum,b_Right_Hippocampus,b_Right_Amygdala,b_Right_Accumbens,cohort,sub,movie
0,0.0,-0.073488,-0.564714,0.429483,0.121324,-0.215214,-0.459373,-0.351671,-1.151885,-0.525397,...,-0.514449,-0.648907,0.011702,-0.482574,0.124356,0.080938,-0.104305,ds002837,10,500daysofsummer
1,1.0,-0.134150,0.223647,0.066747,0.232447,-0.232799,0.244155,-0.193362,0.425376,-0.000460,...,0.120426,-0.472818,-0.125112,-0.228751,0.854094,0.357790,0.124321,ds002837,10,500daysofsummer
2,2.0,-0.035813,0.096488,-0.153626,-0.101278,-0.165800,0.004312,-0.196347,-0.030947,-0.066866,...,0.045549,-0.073201,-0.094871,0.683394,-0.188012,-0.266105,-0.154011,ds002837,10,500daysofsummer
3,3.0,-0.058066,0.114779,-0.189549,-0.126685,-0.174167,0.106635,-0.128238,0.145855,-0.069798,...,-0.137718,-0.162417,-0.210247,0.321874,0.038538,0.092555,0.266721,ds002837,10,500daysofsummer
4,4.0,-0.069651,0.077871,-0.230329,0.145128,-0.108994,0.101450,-0.035268,0.302587,0.056382,...,0.155493,0.013252,-0.163359,-0.030723,-0.063729,0.291210,0.492034,ds002837,10,500daysofsummer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578115,5465.0,-0.108505,-0.189407,0.004995,-0.015330,-0.045463,-0.259948,-0.083452,-0.096849,0.010206,...,-0.004876,0.179838,-0.195015,-0.131687,0.003717,-0.741925,-0.808602,ds002837,9,500daysofsummer
578116,5466.0,-0.073689,0.049798,-0.027798,0.111385,0.152942,-0.059006,-0.002175,-0.088761,0.322320,...,0.252470,0.358804,0.082466,-0.043618,-0.320264,-0.907777,-1.030352,ds002837,9,500daysofsummer
578117,5467.0,-0.027045,-0.112996,0.262119,0.022428,0.224552,-0.016525,0.125966,-0.040568,0.275792,...,0.200092,-0.222411,-0.009208,-0.431448,0.367852,0.155639,-0.080350,ds002837,9,500daysofsummer
578118,5468.0,-0.051507,-0.010687,0.066074,-0.267885,0.024787,-0.008819,0.013349,-0.040666,0.023649,...,-0.223914,-0.465610,-0.191766,0.441294,-0.196498,0.224764,0.890001,ds002837,9,500daysofsummer


## 3. fMRI DFC per interval

In [16]:
df_fmri_path = "brain_states_v2/Outputs/UDFC_Latents_Clusters_300_122.parquet"
df_fmri = pd.read_parquet(folder_path + df_fmri_path)
df_fmri['cohort'] = 'ds002837'
df_fmri.head()

,sub,start,cohort,UDFC_1,UDFC_2,UDFC_3,UDFC_4,UDFC_5,UDFC_6,UDFC_7,...,umap3/5,umap4/5,ThresholdCluster_pca3_8,ThresholdCluster_pca3_27,ThresholdCluster_pca3_125,ThresholdCluster_pca3_512,ThresholdCluster_pca3_2197,ThresholdCluster_pca3_9261,ThresholdCluster_pca3_39304,ThresholdCluster_pca3_166375
0,1,0,ds002837,0.060413,0.196270,0.289575,0.475760,0.183561,0.191599,0.456325,...,-2.398198,-8.640526,4,18,80,387,189,1515,2878,3675
1,1,60,ds002837,0.127351,0.176322,0.259339,0.419744,0.191647,0.145044,0.435915,...,-2.409696,-8.651577,4,18,80,395,1743,1330,2732,3442
2,1,180,ds002837,0.134643,0.104624,0.226564,0.312652,0.149746,0.062600,0.295737,...,-2.405534,-8.646595,4,21,85,403,1773,1357,2606,3451
3,1,300,ds002837,0.136793,0.163141,0.084268,0.376580,0.167698,0.060675,0.202105,...,-2.449202,-8.687985,6,21,90,419,248,1396,2784,3589
4,1,360,ds002837,0.167057,0.170935,0.116738,0.416335,0.252737,0.129300,0.196646,...,-2.448578,-8.687938,6,21,115,411,237,1587,3088,4035


## 4. Static Sub Features

In [16]:
participants = Path('cohorts/ds002837/participants.tsv')
df = pd.read_csv(folder_path / participants, sep="\t")
df["cohort"] = "ds002837"
df["sub"] = df["participant_id"].str.removeprefix("sub-")
df = df.drop(columns=['participant_id'])
df.to_csv(folder_path/output_path/'static_participant_features.csv', index=False)
df

,age,sex,task,cohort,sub
0,23,M,500daysofsummer,ds002837,1
1,25,F,500daysofsummer,ds002837,2
2,23,M,500daysofsummer,ds002837,3
3,23,M,500daysofsummer,ds002837,4
4,22,F,500daysofsummer,ds002837,5
...,...,...,...,...,...
81,50,M,12yearsaslave,ds002837,82
82,18,F,12yearsaslave,ds002837,83
83,22,F,12yearsaslave,ds002837,84
84,23,F,12yearsaslave,ds002837,85


## 5. Fixing time(s) divergence in Brain and Movie dataset

In [13]:
import numpy as np
import pandas as pd

def trim_to_overlap(
    brain: pd.DataFrame,
    stim: pd.DataFrame,
    time_col: str = "time_s",
    movie_col: str = "movie",
    tol_sec: float = 0.0,
):
    """
    Per movie, drop rows whose `time_s` exceeds the last time that BOTH
    tables cover. Head is assumed aligned at start≈0 (true for rounding
    drift); only the divergent tail is removed.

    `tol_sec`: optional slack subtracted from the shared max time, in case
    you want to also shave a safety second beyond the strict overlap.

    Returns (brain_trim, stim_trim). Row counts may still differ if the
    two tables sample at different rates (e.g. 1 s stimulus vs 1.5 s TR);
    that is expected — the AXES now cover the same span, which is what
    the join needs. (Use Option 1 if you also need equal cadence.)
    """
    brain_max = brain.groupby(movie_col)[time_col].max()
    stim_max  = stim.groupby(movie_col)[time_col].max()
    shared_max = pd.concat([brain_max, stim_max], axis=1).min(axis=1)  # per movie

    def _trim(df):
        cap = df[movie_col].map(shared_max) - tol_sec
        return df[df[time_col] <= cap].reset_index(drop=True)

    brain_t = _trim(brain)
    stim_t  = _trim(stim)

    # Report what was cut, per movie (so it is recorded, not silent).
    cut = pd.DataFrame({
        "brain_max": brain_max, "stim_max": stim_max,
        "shared_max": shared_max,
        "brain_drop_s": brain_max - shared_max,
        "stim_drop_s":  stim_max  - shared_max,
    })
    print(cut.round(2).to_string())
    return brain_t, stim_t

In [14]:
brain_t1, stim_t1 = trim_to_overlap(df_all, df_combined)
stim_t1.to_parquet(folder_path/output_path/'known_stimuli_per_second.parquet')
brain_t1.to_parquet(folder_path/output_path/'brain_observed_dynamic_per_second.parquet')

                        brain_max  stim_max  shared_max  brain_drop_s  stim_drop_s
movie                                                                             
12yearsaslave              7714.0      7716      7714.0           0.0          2.0
500daysofsummer            5469.0      5471      5469.0           0.0          2.0
backtothefuture            6673.0      6675      6673.0           0.0          2.0
citizenfour                6803.0      6797      6797.0           6.0          0.0
littlemisssunshine         5899.0      5901      5899.0           0.0          2.0
pulpfiction                8881.0      8882      8881.0           0.0          1.0
split                      6738.0      6740      6738.0           0.0          2.0
theprestige                7514.0      7515      7514.0           0.0          1.0
theshawshankredemption     8180.0      8182      8180.0           0.0          2.0
theusualsuspects           6101.0      6102      6101.0           0.0          1.0


## View

In [7]:
import subprocess, sys
BRAIN_FORECAST_DIR = '/home/tamires/projects/rpp-aevans-ab/tamires/BrainForecast'
sys.path.insert(0, BRAIN_FORECAST_DIR)

In [9]:
import pyarrow.parquet as pq
from brain_forecast.predictors.channel_projection import parse_channels

# All mov_* columns from the stimulus parquet
stim_cols = pq.ParquetFile('/home/tamires/projects/rpp-aevans-ab/tamires/BrainForecast/inputs/known_stimuli_per_second.parquet').schema.names
known = [c for c in stim_cols if c.startswith('mov_')]
print(f"Total mov_* columns: {len(known)}")

# Parse with the default pattern
result = parse_channels(known, proj_dim=64)

print(f"\nDetected {len(result.groups)} multi-dim channels:")
for ch, cols in result.groups.items():
    print(f"  {ch:24s}: {len(cols)} dims -> projects to {min(64, len(cols))}")

print(f"\n{len(result.scalars)} scalar pass-through columns:")
for s in result.scalars:
    print(f"  {s}")

print(f"\nTotal slots into TFT VSN: {len(result.ordered_proj)}")

Total mov_* columns: 4947

Detected 11 multi-dim channels:
  L1                      : 256 dims -> projects to 64
  L2                      : 512 dims -> projects to 64
  L3                      : 1024 dims -> projects to 64
  L4                      : 2048 dims -> projects to 64
  semantic                : 512 dims -> projects to 64
  emotion                 : 7 dims -> projects to 7
  faces                   : 3 dims -> projects to 3
  motion                  : 19 dims -> projects to 19
  audio_mel               : 128 dims -> projects to 64
  audio_spec              : 6 dims -> projects to 6
  narrative               : 384 dims -> projects to 64

48 scalar pass-through columns:
  mov_n_frames
  mov_scene_cut
  mov_surprise_L1
  mov_uncertainty_L1
  mov_surprise_L2
  mov_uncertainty_L2
  mov_surprise_L3
  mov_uncertainty_L3
  mov_surprise_L4
  mov_uncertainty_L4
  mov_surprise_semantic
  mov_uncertainty_semantic
  mov_surprise_motion
  mov_uncertainty_motion
  mov_surprise_emotion
  m

In [10]:
total_raw_accounted = sum(len(cols) for cols in result.groups.values()) + len(result.scalars)
print(f"Raw cols accounted for: {total_raw_accounted}/{len(known)}")
# Should print: 4947/4947

Raw cols accounted for: 4947/4947
